In [1]:
import os
import json
from yt_dlp import YoutubeDL
from pydub import AudioSegment, silence
import whisper
from spleeter.separator import Separator
from spleeter.audio.adapter import AudioAdapter

os.makedirs("data/youtube-audio", exist_ok=True)
os.makedirs("data/youtube-vocal", exist_ok=True)
os.makedirs("data/youtube-text", exist_ok=True)
os.makedirs("data/youtube-audio-chunk", exist_ok=True)

2023-04-14 21:33:31.105001: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
2023-04-14 21:33:35.845822: W tensorflow/compiler/xla/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libnvinfer.so.7'; dlerror: libnvinfer.so.7: cannot open shared object file: No such file or directory; LD_LIBRARY_PATH: /usr/local/cuda/lib64:/usr/local/nccl2/lib:/usr/local/cuda/extras/CUPTI/lib64:/usr/local/cuda/lib64:/usr/local/nccl2/lib:/usr/local/cuda/extras/CUPTI/lib64
2023-04-14 21:33:35.846831: W tensorflow/compiler/xla/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libnvinfer_plugin.so.7'; dlerror: libnvinfer_plugin.so.7: cannot open shared object file: No such file or di

In [ ]:
video_id = "BTe8pFl3oDo"

In [ ]:
# YouTube 動画を wav でダウンロード

ydl_opts = {
    "outtmpl": "data/youtube-audio/%(id)s.%(ext)s",
    "format": "mp3/bestaudio/best",
    "postprocessors": [
        {
            "key": "FFmpegExtractAudio",
            "preferredcodec": "wav",
        }
    ],
}

with YoutubeDL(ydl_opts) as ydl:
    ydl.download(["https://www.youtube.com/watch?v=%s" % video_id])

In [ ]:
# 音声とBGMを分離
# https://github.com/deezer/spleeter/wiki/4.-API-Reference#separator

audio_path = "data/youtube-audio/%s.wav" % video_id
vocal_path = "data/youtube-vocal/"

separator = Separator("spleeter:2stems")
separator.separate_to_file(audio_path, vocal_path)

In [ ]:
# 無音部分でファイルを分割する
# https://github.com/jiaaro/pydub/
# https://github.com/jiaaro/pydub/blob/master/API.markdown#silencesplit_on_silence
#
# file = AudioSegment.from_wav("data/youtube-vocal/%s/vocals.wav" % video_id)
#
# chunks = silence.split_on_silence(
#     file,
#     min_silence_len=3000,
#     silence_thresh=-40,
#     seek_step=1000,
# )
#
# for i, chunk in enumerate(chunks):
#     chunk_path = "data/youtube-audio-chunk/%s_%04d.wav" % (video_id, i + 1)
#     chunk.export(chunk_path, format="wav")

In [3]:
whisper_model = whisper.load_model("medium")

In [8]:
# 音声ファイルの文字起こし

audio_directory = "data/youtube-vocal"
text_directory = "data/youtube-text"

for file in sorted(os.listdir(audio_directory)):
    audio_path = os.path.join(audio_directory, file, "vocals.wav")
    text_path = os.path.join(
        text_directory,
        file.replace(".wav", "") + ".json",
    )

    result = whisper_model.transcribe(audio_path, verbose=True, language="ja")

    with open(text_path, "w") as outfile:
        json.dump(result, outfile, indent=4, ensure_ascii=False)


[00:00.000 --> 00:02.000] ご視聴ありがとうございました
[00:30.000 --> 00:32.000] ご視聴ありがとうございました
[01:00.020 --> 01:02.020] ここですぐ節制させてみたいと思います
[01:04.960 --> 01:06.160] はいっ
[01:08.880 --> 01:18.880] こういう感じもしないと
[01:18.840 --> 01:21.760] うぅぅぅ
[01:24.740 --> 01:15.120] ここで Hastattmotter
[01:25.560 --> 01:27.560] この配信は、 smaller risesが自由に話し
[01:27.560 --> 01:29.560] ただの期待はしないでください
[01:36.560 --> 01:39.560] こんばんねークロー
[01:45.560 --> 01:47.560] じゃねーんだよなー
[01:47.560 --> 01:49.560] お前は何だよ
[01:55.560 --> 01:57.560] おいおいおいおい
[01:58.560 --> 02:00.560] 青木里高校さんさ
[02:01.560 --> 02:03.560] 前世との中で
[02:04.560 --> 02:06.560] 唯一3Dなんですけど
[02:08.560 --> 02:10.560] それにさ
[02:10.560 --> 02:12.560] 3D学役ですってなってから
[02:12.560 --> 02:14.560] 何日経ったと思う
[02:14.560 --> 02:16.560] ちょっと振り返ってみよっか
[02:18.560 --> 02:20.560] みんなも気になるかなって思うんで
[02:25.560 --> 02:27.560] こちらの配信覚えてますかね
[02:28.560 --> 02:30.560] いつやったと思うみんな
[02:32.560 --> 02:34.560] さすがにね日にちまで覚えてるやついないかなと思うんでね
[02:35.560 --> 02:37.560] 見ていきますかね
[02:37.560 --> 02:41.560] えーっと日にちが


In [19]:
# whisper で文字起こしした結果の JSON ファイルを読み取って、セグメントごとに音声ファイルを分割する

text_directory = "data/youtube-text"

for filename in sorted(os.listdir(text_directory)):
    with open(os.path.join(text_directory, filename), "r") as file:
        data = json.load(file)

    for segment in data["segments"]:
        video_id = filename.replace(".json", "")
        input_file = "data/youtube-vocal/%s/vocals.wav" % video_id
        output_file = "data/youtube-audio-chunk/%s_%04d.wav" % (
            video_id,
            segment["id"],
        )

        audio = AudioSegment.from_wav(input_file)
        trimmed_audio = audio[segment["start"] * 1000: segment["end"] * 1000]

        trimmed_audio.export(output_file, format="wav")

        text = segment["text"]
        print(f"{output_file}: {text}")


data/youtube-audio-chunk/BTe8pFl3oDo_0000.wav: ご視聴ありがとうございました
data/youtube-audio-chunk/BTe8pFl3oDo_0001.wav: ご視聴ありがとうございました
data/youtube-audio-chunk/BTe8pFl3oDo_0002.wav: ここですぐ節制させてみたいと思います
data/youtube-audio-chunk/BTe8pFl3oDo_0003.wav: はいっ
data/youtube-audio-chunk/BTe8pFl3oDo_0004.wav: こういう感じもしないと
data/youtube-audio-chunk/BTe8pFl3oDo_0005.wav: うぅぅぅ
data/youtube-audio-chunk/BTe8pFl3oDo_0006.wav: ここで Hastattmotter
data/youtube-audio-chunk/BTe8pFl3oDo_0007.wav: この配信は、 smaller risesが自由に話し
data/youtube-audio-chunk/BTe8pFl3oDo_0008.wav: ただの期待はしないでください
data/youtube-audio-chunk/BTe8pFl3oDo_0009.wav: こんばんねークロー
data/youtube-audio-chunk/BTe8pFl3oDo_0010.wav: じゃねーんだよなー
data/youtube-audio-chunk/BTe8pFl3oDo_0011.wav: お前は何だよ
data/youtube-audio-chunk/BTe8pFl3oDo_0012.wav: おいおいおいおい
data/youtube-audio-chunk/BTe8pFl3oDo_0013.wav: 青木里高校さんさ
data/youtube-audio-chunk/BTe8pFl3oDo_0014.wav: 前世との中で
data/youtube-audio-chunk/BTe8pFl3oDo_0015.wav: 唯一3Dなんですけど
data/youtube-audio-chunk/BTe8pFl3oDo_0016.wav: それにさ
dat